In [1]:
import torch
import torch.optim as optim
import numpy as np

In [2]:
device = torch.device("cuda")

# Load clip2styles Matrix

In [3]:
clip2styles_matrix = np.load("clip_data/clip2styles.npy")

# Define Clip2Style Mapper

In [7]:
class Clip2StyleMapper(torch.nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dim=1024, num_layers=3):
        super().__init__()
        layers = []
        current_dim = in_dim
        
        for i in range(num_layers):
            next_dim = out_dim if i == num_layers - 1 else hidden_dim
            
            layers.extend([torch.nn.Linear(current_dim, next_dim)])
            current_dim = next_dim
            
        self.net = torch.nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
# 1. Setup Data and Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Clip2StyleMapper(in_dim=512, out_dim=6048).to(device)
learning_rate = 1e-5
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.MSELoss()

# clip2styles_matrix shape: (6048, 512)
matrix_gt = torch.from_numpy(clip2styles_matrix).float().to(device)

# 2. Training Loop
epochs = 50000
batch_size = 64

for epoch in range(epochs):
    # Generate synthetic input (CLIP direction)
    inputs = torch.randn(batch_size, 512).to(device)
    
    # Calculate ground truth: (batch, 6048)
    with torch.no_grad():
        targets = inputs @ matrix_gt.t()

    # Forward pass
    outputs = model(inputs)
    loss = criterion(outputs, targets)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch [{epoch}/{epochs}], Loss: {loss.item():.6f}")

Epoch [0/50000], Loss: 1.069944
Epoch [50/50000], Loss: 1.019753
Epoch [100/50000], Loss: 1.001322
Epoch [150/50000], Loss: 1.019027
Epoch [200/50000], Loss: 0.988969
Epoch [250/50000], Loss: 0.965822
Epoch [300/50000], Loss: 0.958546
Epoch [350/50000], Loss: 1.014254
Epoch [400/50000], Loss: 0.956129
Epoch [450/50000], Loss: 0.976845
Epoch [500/50000], Loss: 0.957831
Epoch [550/50000], Loss: 0.947259
Epoch [600/50000], Loss: 0.889268
Epoch [650/50000], Loss: 0.869955
Epoch [700/50000], Loss: 0.834022
Epoch [750/50000], Loss: 0.769726
Epoch [800/50000], Loss: 0.735148
Epoch [850/50000], Loss: 0.744914
Epoch [900/50000], Loss: 0.685947
Epoch [950/50000], Loss: 0.645064
Epoch [1000/50000], Loss: 0.651146
Epoch [1050/50000], Loss: 0.605091
Epoch [1100/50000], Loss: 0.557064
Epoch [1150/50000], Loss: 0.545659
Epoch [1200/50000], Loss: 0.515356
Epoch [1250/50000], Loss: 0.507159
Epoch [1300/50000], Loss: 0.486096
Epoch [1350/50000], Loss: 0.452414
Epoch [1400/50000], Loss: 0.464024
Epoch [1

In [ ]:
torch.save(model.state_dict(), 'out/mapping_network.pth')